# P4B execution package validation gate

This notebook is **validation-only**. It must not execute a scientific task. Keep `P4B_AUTHORIZED=false`, `RESULT_BEARING=false`, and `SCIENTIFIC_EXECUTION=0`.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
print('DRIVE_MOUNT=PASS')


In [ ]:
AUTHORIZED_COMMIT = '<SEPARATELY_REVIEWED_EXACT_P4B_COMMIT_SHA>'
REPO = 'ArashSalehpourac/BreastCancer-Imbalance-Benchmark_35'
assert len(AUTHORIZED_COMMIT) == 40 and all(c in '0123456789abcdef' for c in AUTHORIZED_COMMIT)
print('P4B_EXACT_COMMIT_FORMAT=PASS')


In [ ]:
import os, pathlib, subprocess
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('GITHUB_TOKEN Colab secret is required')
repo_root = pathlib.Path('/content/BreastCancer-Imbalance-Benchmark_35')
if repo_root.exists():
    subprocess.run(['rm','-rf',str(repo_root)], check=True)
askpass = pathlib.Path('/content/p4b_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "x-access-token" ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
askpass.chmod(0o700)
env = dict(os.environ)
env['GITHUB_TOKEN'] = token
env['GIT_ASKPASS'] = str(askpass)
env['GIT_TERMINAL_PROMPT'] = '0'
subprocess.run(['git','clone',f'https://github.com/{REPO}.git',str(repo_root)], check=True, env=env)
subprocess.run(['git','-C',str(repo_root),'checkout','--detach',AUTHORIZED_COMMIT], check=True)
actual = subprocess.check_output(['git','-C',str(repo_root),'rev-parse','HEAD'], text=True).strip()
if actual != AUTHORIZED_COMMIT:
    raise RuntimeError(f'Git identity mismatch: {actual}')
print('P4B_REPOSITORY_IDENTITY=PASS')


In [ ]:
import sys, subprocess
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'P4B_PYTHON_VERSION_GATE=FAIL: {sys.version}')
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo_root/'requirements/p3-preflight.txt')], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root),'--no-deps'], check=True)
print('P4B_ENVIRONMENT_INSTALL=PASS')


In [ ]:
completed = subprocess.run([
    sys.executable, str(repo_root/'scripts/p4b_execute.py'),
    '--validate-only', '--repo-root', str(repo_root),
    '--expected-git-commit', AUTHORIZED_COMMIT,
    '--allow-review-head'
], check=True, text=True, capture_output=True)
print(completed.stdout)
required = [
    'P4B_PACKAGE_VALIDATION=PASS',
    'P4B_FIRST_RUN_MAX_TASKS=1',
    'P4B_AUTHORIZED=false',
    'RESULT_BEARING=false',
    'SCIENTIFIC_EXECUTION=0',
]
for gate in required:
    if gate not in completed.stdout.splitlines():
        raise RuntimeError(f'missing gate: {gate}')
print('P4B_COLAB_PACKAGE_VALIDATION=PASS')
print('P4B_AUTHORIZED=false')
print('RESULT_BEARING=false')
print('SCIENTIFIC_EXECUTION=0')
